In [25]:
from agentlib.search import KnowledgeBase
from dotenv import load_dotenv
from pydantic_ai import Agent
from typing import List, Any
from IPython.display import Markdown

from pydantic_ai.messages import ModelMessagesTypeAdapter
from pydantic import BaseModel
import json
import secrets
from pathlib import Path
from datetime import datetime
import random
from tqdm.auto import tqdm
import pandas as pd

load_dotenv()

True

In [2]:
kb = KnowledgeBase("google", "adk-python")
kb.build()

# query = "How do I run an agent locally using the ADK?"
# results = kb.query(query, num_results=5)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

In [4]:
docs = kb.get_docs()

In [5]:
docs[0]

{'name': 'Bug report',
 'about': 'Create a report to help us improve',
 'title': '',
 'labels': '',
 'assignees': '',
 'content': "## 🔴 Required Information\n*Please ensure all items in this section are completed to allow for efficient\ntriaging. Requests without complete information may be rejected / deprioritized.\nIf an item is not applicable to you - please mark it as N/A*\n\n**Describe the Bug:**\nA clear and concise description of what the bug is.\n\n**Steps to Reproduce:**\nPlease provide a numbered list of steps to reproduce the behavior:\n1. Install '...'\n2. Run '....'\n3. Open '....'\n4. Provide error or stacktrace\n\n**Expected Behavior:**\nA clear and concise description of what you expected to happen.\n\n**Observed Behavior:**\nWhat actually happened? Include error messages or crash stack traces here.\n\n**Environment Details:**\n\n - ADK Library Version (pip show google-adk):\n - Desktop OS:** [e.g., macOS, Linux, Windows]\n - Python Version (python -V):\n\n**Model Infor

In [57]:
system_prompt = """
You are adk_agent, an intelligent assistant that answers user questions strictly using the content of the Google ADK Python repository.

---------------------
🔍 RETRIEVAL RULES
---------------------
• Always call the `text_search` tool before answering.
• Pass the user's full query to the tool.
• Read the returned sections carefully.
• Base your answer ONLY on those retrieved chunks.
• Do not use outside knowledge.

If no relevant information is returned:
→ Say: “I couldn’t find relevant information in the documentation,” and offer general guidance.

---------------------
📘 ANSWERING RULES
---------------------
• Provide clear, concise, accurate explanations.
• Rephrase in your own words unless quoting is necessary.
• Do not invent APIs, functions, or features not found in retrieved documents.
• If multiple chunks apply, merge them into a coherent answer.

---------------------
📎 REFERENCES (MANDATORY)
---------------------
After answering, include references to the source files used.

Reference format:
[FILE NAME](https://github.com/google/adk-python/blob/main/<PATH>/<FILENAME>)

Rules:
• Use ONLY filenames from the retrieved search results.
• Convert plain filenames to GitHub URLs so that users can click to verify.
• If multiple files contributed, list all of them.

Sources:
- [agents.md](https://github.com/google/adk-python/blob/main/docs/agents.md)
- [tools.md](https://github.com/google/adk-python/blob/main/docs/tools.md)

---------------------
🎯 MISSION
---------------------
Help users understand the ADK Python framework accurately.
Always search first, answer second, and cite sources.
"""

In [7]:
def text_search(query: str) -> List[Any]:
    """
    Perform hybrid search (text + vector) on the knowledge base.
    Returns a list of documents relevant to the query.
    """
    return kb.query(query, num_results=5)

In [8]:
agent = Agent(
    name="adk_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='openai:gpt-4o-mini'
)


In [6]:
# result = await agent.run(user_prompt=query)

In [4]:
# Markdown(result.response.parts[0].content)

In [9]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []
    
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
    
            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']
    
            parts.append(part)
    
        message = {
            'kind': m['kind'],
            'timestamp': m.get('timestamp'),
            'parts': parts
        }
    
        log_simplified.append(message)
    return log_simplified

def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": simplify_log_messages(dict_messages),
        "source": source
    }

LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def normalize_timestamp(ts):
    if isinstance(ts, datetime):
        return ts
    if isinstance(ts, dict) and "timestamp" in ts:
        ts = ts["timestamp"]
    if isinstance(ts, str):
        return datetime.fromisoformat(ts.replace("Z", "+00:00"))

    raise TypeError(f"Unsupported timestamp type: {type(ts)}")

def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts_raw = entry['messages'][-1]['timestamp']
    ts_obj = normalize_timestamp(ts_raw)
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

In [15]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
messages = result.new_messages()
log_interaction_to_file(agent, messages)

 How do I run an agent locally using the ADK?


To run an agent locally using the ADK (Agent Development Kit), you should follow these steps:

1. **Define Your Agent**: First, you need to define your agent using the ADK framework. For example:

    ```python
    from google.adk.agents import Agent
    from google.adk.tools import google_search

    root_agent = Agent(
        name="search_assistant",
        model="gemini-2.5-flash",
        instruction="You are a helpful assistant. Answer user questions using Google Search when needed.",
        description="An assistant that can search the web.",
        tools=[google_search]
    )
    ```

2. **Install ADK**: If you haven't installed ADK, you can do so using pip:

    ```bash
    pip install google-adk
    ```

3. **Run the Agent**: Once your agent is defined, you can run it using a command in your terminal:

    ```bash
    adk run .
    ```
   or to run a web interface:
   
   ```bash
   adk web
   ```

4. **Interact with the Agent**: After running the agent, you can interact w

PosixPath('logs/adk_agent_20260328_173415_9765bf.json')

In [10]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [11]:
class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

In [12]:
eval_agent = Agent(
    name='eval_agent',
    model='openai:gpt-5-nano',
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)


In [13]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [14]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log = json.dumps(messages)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output 


log_record = load_log_file('./logs/adk_agent_20260328_173415_9765bf.json')
eval1 = await evaluate_log_record(eval_agent, log_record)

In [36]:
for check in eval1.checklist:
    print(check)

check_name='instructions_follow' justification='The assistant used the retrieval/answering flow as instructed and prepared to present an answer derived from the ADK docs.' check_pass=True
check_name='instructions_avoid' justification='No external knowledge beyond the retrieved ADK documentation was added.' check_pass=True
check_name='answer_relevant' justification='The answer directly addresses how to run an agent locally using the ADK.' check_pass=True
check_name='answer_clear' justification='Steps are listed in order with optional code blocks and commands.' check_pass=True
check_name='answer_citations' justification='Source files are cited with GitHub URLs to the retrieved docs.' check_pass=True
check_name='completeness' justification='Includes agent definition, ADK installation, running, interaction, activation, and references.' check_pass=True
check_name='tool_call_search' justification='A text search/retrieval step was performed prior to answering (per retrieval rules).' check_pas

In [26]:
question_generation_prompt = """
You are the question_generator agent.
Your task is to create a list of user questions that will be used to evaluate another agent called “adk_agent”.

---------------------
🎯 YOUR TASK
---------------------
Generate a list of **diverse, realistic, challenging user questions** that a developer might ask about the Google ADK Python repositor byased on the provided content.

Requirements:
• Include easy, medium, and advanced level questions.

Output format:
A JSON object with a single field:
{
  "questions": [ "question 1", "question 2", ... ]
}

Keep questions short, concrete, and specific.
Generate one question for each record.
"""

In [16]:
class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='openai:gpt-4o-mini',
    output_type=QuestionsList
)


In [17]:
sample = random.sample(docs, 10)
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

In [19]:
questions

['What are the key differences between static and dynamic instructions in an ADK agent?',
 'How do you set up API credentials for the Bingo Digital Pet agent?',
 'Can you explain the process of feeding Bingo and how it affects his hunger states?',
 'What is the purpose of the session state in the session state agent?',
 'How can I list table names in a GCP Spanner database using the Spanner tools?',
 "What steps should I take if I encounter an 'Unauthorized' error while running the Notion MCP agent?",
 'How does the A2A OAuth Authentication sample agent manage multiple agents and OAuth workflows?',
 'What is the significance of the `VertexAiCodeExecutor` for executing Python code?',
 'Could you explain the human-in-the-loop workflow implemented in the A2A Human-in-the-Loop sample?',
 'How does the Files Retrieval agent use the `gemini-embedding-2-preview` model for retrieval-augmented generation?',
 'What formats are supported for passing credentials in the Spanner tools agent?',
 'How

In [21]:
for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/12 [00:00<?, ?it/s]

What are the key differences between static and dynamic instructions in an ADK agent?
In the context of an ADK agent, the key differences between static and dynamic instructions revolve around how the instructions are processed and utilized.

1. **Static Instructions**:
   - These are defined at agent initialization, meaning they do not change during the agent's execution. Static instructions are processed once, and any necessary content is referenced upfront in a way that is resolute during initialization.
   - For example, static instruction may include mixed content that can consist of inline data or references to external files, ensuring that all content is available for the agent to use consistently.

2. **Dynamic Instructions**:
   - In contrast, dynamic instructions are generated or modified during the agent's operation. They can adapt based on the context or inputs received at runtime.
   - This allows for more flexibility, as the responses or actions of the agent can vary acco

In [22]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'adk_agent' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)

In [23]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

  0%|          | 0/12 [00:00<?, ?it/s]

In [53]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: bool(c.check_pass) for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

In [54]:
df_evals = pd.DataFrame(rows)
bool_cols = df_evals.columns.difference(['file', 'question', 'answer', 'example'])
df_evals[bool_cols] = df_evals[bool_cols].astype('boolean')

In [55]:
df_evals.mean(numeric_only=True)

instructions_follow    0.909091
instructions_avoid     0.909091
answer_relevant             0.9
answer_clear                0.9
answer_citations            0.9
completeness                0.9
tool_call_search       0.909091
dtype: Float64